Goal: map CUIs to NER classes [DISEASE, MEDICATION, PROCEDURE, SYMPTOM]

Steps:
* load CUI-TUI map
* map CUI to TUI
* map TUI to NER class

In [ ]:
import os
import pandas as pd

In [ ]:
sem_groups = pd.read_csv('/media/bramiozo/Storage1/bramiozo/DEV/GIT/UMCU/SCUCE/assets/SemGroups.txt', sep='|', 
                         header=None, names=['sem_group', 'sem_type', 'tui', 'description'])

In [ ]:
sem_groups[sem_groups.description.str.lower().str.contains('procedure')]

In [175]:
sem_groups.description.unique().tolist()

['Activity',
 'Behavior',
 'Daily or Recreational Activity',
 'Event',
 'Governmental or Regulatory Activity',
 'Individual Behavior',
 'Machine Activity',
 'Occupational Activity',
 'Social Behavior',
 'Anatomical Structure',
 'Body Location or Region',
 'Body Part, Organ, or Organ Component',
 'Body Space or Junction',
 'Body Substance',
 'Body System',
 'Cell',
 'Cell Component',
 'Embryonic Structure',
 'Fully Formed Anatomical Structure',
 'Tissue',
 'Amino Acid, Peptide, or Protein',
 'Antibiotic',
 'Biologically Active Substance',
 'Biomedical or Dental Material',
 'Chemical',
 'Chemical Viewed Functionally',
 'Chemical Viewed Structurally',
 'Clinical Drug',
 'Element, Ion, or Isotope',
 'Enzyme',
 'Hazardous or Poisonous Substance',
 'Hormone',
 'Immunologic Factor',
 'Indicator, Reagent, or Diagnostic Aid',
 'Inorganic Chemical',
 'Nucleic Acid, Nucleoside, or Nucleotide',
 'Organic Chemical',
 'Pharmacologic Substance',
 'Receptor',
 'Vitamin',
 'Classification',
 'Conceptua

In [192]:
sem_groups.query('description == "Acquired Abnormality"') 

,sem_group,sem_type,tui,description
55,DISO,Disorders,T020,Acquired Abnormality


In [201]:
TUI_MAP = {   
'SYMPTOM': {
    'T184': 'SYMPTOM',
    'T033': 'SYMPTOM',
    'T048': 'SYMPTOM',
    'T201': 'SYMPTOM',
    'T034': 'SYMPTOM',
}, 
'DISEASE': {
    'T047': 'DISEASE',
    'T020': 'DISEASE',
    'T190': 'DISEASE',
    'T049': 'DISEASE',
    'T019': 'DISEASE',
    'T033': 'DISEASE',
    'T037': 'DISEASE',
    'T048': 'DISEASE',
    'T191': 'DISEASE',
    'T046': 'DISEASE',
    'T050': 'DISEASE',
},
 'MEDICATION': {
    'T200': 'MEDICATION',
    'T195': 'MEDICATION',
    'T121': 'MEDICATION',
    'T125': 'MEDICATION',
    'T127': 'MEDICATION',
 },
  'PROCEDURE': {
    'T060': 'PROCEDURE',
    'T059': 'PROCEDURE',
    'T061': 'PROCEDURE',
    'T065': 'PROCEDURE',
    'T058': 'PROCEDURE',
    'T063': 'PROCEDURE',
    'T062': 'PROCEDURE',
    'T034': 'PROCEDURE',
  }
} # METAMAP

In [ ]:
cui_to_tui = pd.read_parquet('/media/bramiozo/Storage2/DATA/ONTOLOGY/UMLS/MAPPINGS/CUI_TUI.parquet')

In [ ]:
cui_to_def = pd.read_parquet('/media/bramiozo/Storage2/DATA/ONTOLOGY/UMLS/MAPPINGS/CUI_DEF.parquet')

In [ ]:
cui_to_def = cui_to_def.groupby('CUI').STR.apply(lambda x: ','.join(x)).reset_index()

In [202]:
medmentions_gpt_translated = pd.read_csv('/media/bramiozo/Storage2/DATA/CORPORA/NL/NER_L/DutchClinicalCorpora-main/MedMentions/GPT_translated/MedMentions_Dutch_GPT_translated_annotations.csv', sep=',')
medmentions_google_translated = pd.read_csv('/media/bramiozo/Storage2/DATA/CORPORA/NL/NER_L/DutchClinicalCorpora-main/MedMentions/Google_translated/MedMentions_Dutch_Google_translated_annotations.csv', sep=',')

In [ ]:
cui_to_tui = pd.merge(cui_to_tui, sem_groups[['tui', 'description']], how='left',
                        left_on='TUI', right_on='tui').drop('STY', axis=1).rename(columns={'description': 'GROUP_DESC'})

In [ ]:
gpt_medmention_cuis = medmentions_gpt_translated.CUI.unique()
google_medmention_cuis = medmentions_google_translated.CUI.unique()

In [ ]:
cui_to_tui = cui_to_tui.groupby('CUI').TUI.unique().reset_index('CUI')

In [203]:
medmentions_gpt_translated = pd.merge(medmentions_gpt_translated, cui_to_tui, how='left', left_on='CUI', right_on='CUI')

In [204]:
medmentions_google_translated = pd.merge(medmentions_google_translated, cui_to_tui, how='left', left_on='CUI', right_on='CUI')

In [205]:
medmentions_gpt_translated.dropna(subset=['TUI'], inplace=True)
medmentions_google_translated.dropna(subset=['TUI'], inplace=True)

In [206]:
unfolded_multi_tui_list= []
for row in medmentions_gpt_translated.itertuples():
    if len(row.TUI) > 1:
        for tui in row.TUI:
            new_row = row._asdict()
            new_row['TUI'] = tui
            unfolded_multi_tui_list.append(new_row)
unfolded_multi_tui_dataframe = pd.DataFrame(unfolded_multi_tui_list)
unfolded_multi_tui_dataframe.drop(columns=['Index'], inplace=True)
unfolded_multi_tui_dataframe.rename(columns={'_3': 'from'}, inplace=True)

mii = medmentions_gpt_translated\
                .loc[medmentions_gpt_translated.TUI.apply(lambda x: len(x))>1, 'TUI'].index

medmentions_gpt_translated = medmentions_gpt_translated.loc[~medmentions_gpt_translated.index.isin(mii)]

medmentions_gpt_translated\
        .loc[medmentions_gpt_translated.TUI.apply(lambda x: len(x))==1, 'TUI'] =\
    medmentions_gpt_translated.loc[medmentions_gpt_translated.TUI.apply(lambda x: len(x))==1, 'TUI'].apply(lambda x: x[0])

medmentions_gpt_translated = pd.concat([medmentions_gpt_translated, unfolded_multi_tui_dataframe], ignore_index=True)

In [207]:
unfolded_multi_tui_list= []
for row in medmentions_google_translated.itertuples():
    if len(row.TUI) > 1:
        for tui in row.TUI:
            new_row = row._asdict()
            new_row['TUI'] = tui
            unfolded_multi_tui_list.append(new_row)
unfolded_multi_tui_dataframe = pd.DataFrame(unfolded_multi_tui_list)
unfolded_multi_tui_dataframe.drop(columns=['Index'], inplace=True)
unfolded_multi_tui_dataframe.rename(columns={'_3': 'from'}, inplace=True)

mii = medmentions_google_translated\
                .loc[medmentions_google_translated.TUI.apply(lambda x: len(x))>1, 'TUI'].index

medmentions_google_translated = medmentions_google_translated.loc[~medmentions_google_translated.index.isin(mii)]

medmentions_google_translated\
        .loc[medmentions_google_translated.TUI.apply(lambda x: len(x))==1, 'TUI'] =\
    medmentions_google_translated.loc[medmentions_google_translated.TUI.apply(lambda x: len(x))==1, 'TUI'].apply(lambda x: x[0])

medmentions_google_translated = pd.concat([medmentions_google_translated, unfolded_multi_tui_dataframe], ignore_index=True)

In [209]:
medmentions_google_translated['SYMPTOM'] = medmentions_google_translated.TUI.map(TUI_MAP['SYMPTOM'])
medmentions_gpt_translated['SYMPTOM'] = medmentions_gpt_translated.TUI.map(TUI_MAP['SYMPTOM'])

medmentions_google_translated['DISEASE'] = medmentions_google_translated.TUI.map(TUI_MAP['DISEASE'])
medmentions_gpt_translated['DISEASE'] = medmentions_gpt_translated.TUI.map(TUI_MAP['DISEASE'])

medmentions_google_translated['MEDICATION'] = medmentions_google_translated.TUI.map(TUI_MAP['MEDICATION'])
medmentions_gpt_translated['MEDICATION'] = medmentions_gpt_translated.TUI.map(TUI_MAP['MEDICATION'])

medmentions_google_translated['PROCEDURE'] = medmentions_google_translated.TUI.map(TUI_MAP['PROCEDURE'])
medmentions_gpt_translated['PROCEDURE'] = medmentions_gpt_translated.TUI.map(TUI_MAP['PROCEDURE'])


In [211]:
medmentions_google_translated = medmentions_google_translated.dropna(subset=['SYMPTOM', 'DISEASE', 'MEDICATION', 'PROCEDURE'], how='all')
medmentions_gpt_translated = medmentions_gpt_translated.dropna(subset=['SYMPTOM', 'DISEASE', 'MEDICATION', 'PROCEDURE'], how='all')

In [ ]:
medmentions_google_translated = medmentions_google_translated.melt(id_vars=['id', 'CUI', 'from', 'to', 'value', 'TUI'],
                                    value_vars=['SYMPTOM', 'DISEASE', 'PROCEDURE', 'MEDICATION'],
                                    value_name='ENTITY_GROUP').dropna(subset='ENTITY_GROUP').drop('variable', axis=1)

In [232]:
medmentions_gpt_translated = medmentions_gpt_translated.melt(id_vars=['id', 'CUI', 'from', 'to', 'value', 'TUI'],
                                    value_vars=['SYMPTOM', 'DISEASE', 'PROCEDURE', 'MEDICATION'],
                                    value_name='ENTITY_GROUP').dropna(subset='ENTITY_GROUP').drop('variable', axis=1)

In [236]:
medmentions_google_translated = medmentions_google_translated.rename(columns={'value': 'text', 
                                                                              'from': 'start_span', 
                                                                              'to': 'end_span', 
                                                                              'ENTITY_GROUP': 'label'})

medmentions_gpt_translated = medmentions_gpt_translated.rename(columns={'value': 'text', 
                                                                              'from': 'start_span', 
                                                                              'to': 'end_span', 
                                                                              'ENTITY_GROUP': 'label'})

In [242]:
medmentions_gpt_translated[['id', 'start_span', 'end_span', 'text', 'label']].reset_index(drop=True)\
    .to_csv('/media/bramiozo/Storage2/DATA/CORPORA/NL/NER_L/DutchClinicalCorpora-main/MedMentions/GPT_translated/medner_medmentions_gpt.csv',
            index=False)

medmentions_google_translated[['id', 'start_span', 'end_span', 'text', 'label']].reset_index(drop=True)\
    .to_csv('/media/bramiozo/Storage2/DATA/CORPORA/NL/NER_L/DutchClinicalCorpora-main/MedMentions/Google_translated/medner_medmentions_google.csv',
            index=False)